# Student Performance — Notebook

Generated by `scripts/build_notebook.py` and executed end to end.

1. Load both subject files and validate them
2. Audit the cleaning decisions (duplicates, zero grades)
3. Pair maths and Portuguese students and test the difference
4. Compare the three grade-availability scenarios


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from student import config, data
from student.clean import clean
from student.merge import compare_subjects, pair_students, pairing_report

pd.set_option("display.max_columns", 50)


## 1. Load and validate

In [2]:
frames = data.load_all()
combined = data.combine(frames)
validation = data.validate(combined)
print(f"rows={validation['rows']:,} zero G3={validation['zero_grades']}")
print("pass rate by subject:", validation["pass_rate"])
combined.groupby("subject")[["G1", "G2", "G3"]].mean().round(2)


[data] Mathematics  395 students
[data] Portuguese   649 students
[data] combined 1,044 rows
rows=1,044 zero G3=53
pass rate by subject: {'mat': 0.671, 'por': 0.846}


,G1,G2,G3
subject,,,
mat,10.91,10.71,10.42
por,11.40,11.57,11.91


## 2. Cleaning audit

In [3]:
cleaned, log = clean(combined)
print("duplicates kept:", log["duplicates_kept"])
print("duplicate pairing keys:", log["duplicate_merge_keys"])
print("duplicate policy:", log["duplicate_policy"]["drop"], "-",
      log["duplicate_policy"]["reason"])
pd.DataFrame(log["by_subject"]).T


duplicates kept: 0
duplicate pairing keys: 382
duplicate policy: False - identical attribute vectors can be different students; pairing multiplicity is handled in merge.py


,rows,mean_g3,median_g3,pass_rate
mat,395.0,10.415,11.0,0.6709
por,649.0,11.906,12.0,0.8459


### The merge-key trap

`student-merge.R` in the archive pairs on 13 attributes and reports 382 students.
Joining on all shared non-grade columns instead looks safer but collapses the match set.


In [4]:
report = pairing_report(frames["mat"], frames["por"])
pd.Series({
    "documented keys": len(report["keys_used"]),
    "matched pairs": report["pairs"],
    "all shared columns": report["all_shared_columns_tried"],
    "rows from naive merge": report["naive_merge_rows"],
})


documented keys           13
matched pairs            382
all shared columns        30
rows from naive merge     39
dtype: int64

## 3. Paired comparison

In [5]:
pairs = pair_students(frames["mat"], frames["por"])
comparison = compare_subjects(pairs)
pd.Series({
    "pairs": comparison["pairs"],
    "maths mean": comparison["mean_mat"],
    "portuguese mean": comparison["mean_por"],
    "mean delta": comparison["mean_delta"],
    "improved": comparison["improved"],
    "worsened": comparison["worsened"],
    "tied": comparison["tied"],
    "sign test p": comparison["sign_test_p"],
    "wilcoxon p": comparison["wilcoxon_p"],
    "grade correlation": comparison["grade_correlation"],
})


pairs                3.820000e+02
maths mean           1.038700e+01
portuguese mean      1.251600e+01
mean delta           2.128000e+00
improved             2.490000e+02
worsened             9.500000e+01
tied                 3.800000e+01
sign test p          4.705118e-17
wilcoxon p           8.676252e-21
grade correlation    4.803000e-01
dtype: float64

## 4. Leakage scenarios

Read from `reports/metrics.json`.

In [6]:
import json

path = ROOT / "reports" / "metrics.json"
metrics = json.loads(path.read_text()) if path.exists() else {}

if metrics:
    display(pd.DataFrame(metrics["leakage_table"]).set_index("scenario").round(3))
    display(pd.DataFrame(metrics["cross_subject_transfer"]).T)
    display(pd.DataFrame(metrics["feature_importance"]).round(4))
else:
    print("Run `python run.py train` first.")


,regression_r2,regression_rmse,classification_auc,classification_f1
scenario,,,,
no_grades,0.224,3.393,0.766,0.861
with_g1,0.843,1.527,0.943,0.948
with_g1_g2,0.876,1.357,0.958,0.927


,train_subject,test_subject,regression,classification
mat_to_por,mat,por,"{'rmse': 3.6858, 'mae': 2.8129, 'r2': -0.3036}","{'roc_auc': 0.6913, 'accuracy': 0.7227, 'preci..."
por_to_mat,por,mat,"{'rmse': 4.3415, 'mae': 3.1006, 'r2': 0.0997}","{'roc_auc': 0.6156, 'accuracy': 0.6962, 'preci..."


,feature,importance
0,failures,0.2006
1,absences,0.0761
2,absence_flag,0.0518
3,study_balance,0.0334
4,parent_education,0.0321
5,health,0.0272
6,age,0.0256
7,social_index,0.0254
8,subject_por,0.0252
9,alcohol_index,0.0239


## 5. Reading the result

The high-R² model uses grades from the same course, so it cannot run before those grades exist. The reported early-warning model is the `no_grades` scenario.